# Potongan.id — Colab Backend (T4 GPU)

Backend **Potongan.id** berjalan di GPU T4 Colab. Hasil klip + database tersimpan otomatis di Google Drive (`MyDrive/AutoClipperData`).

**Urutan:** jalankan semua sel berurutan. Pastikan **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# 1. Mount Google Drive (hasil klip + history.db tersimpan di sini)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. System deps: FFmpeg, cloudflared, + font subtitle (agar libass tidak fallback)
!apt-get update -qq
!apt-get install -y -qq ffmpeg fontconfig
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import os
FONT_DIR = '/usr/share/fonts/truetype/potongan'
os.makedirs(FONT_DIR, exist_ok=True)
FONTS = {
    'Anton': 'https://github.com/google/fonts/raw/main/ofl/anton/Anton-Regular.ttf',
    'BebasNeue': 'https://github.com/google/fonts/raw/main/ofl/bebasneue/BebasNeue-Regular.ttf',
    'Montserrat': 'https://github.com/google/fonts/raw/main/ofl/montserrat/Montserrat%5Bwght%5D.ttf',
    'Oswald': 'https://github.com/google/fonts/raw/main/ofl/oswald/Oswald%5Bwght%5D.ttf',
    'Poppins': 'https://github.com/google/fonts/raw/main/ofl/poppins/Poppins-Bold.ttf',
    'PermanentMarker': 'https://github.com/google/fonts/raw/main/ofl/permanentmarker/PermanentMarker-Regular.ttf',
}
for name, url in FONTS.items():
    !wget -q -O "{FONT_DIR}/{name}.ttf" "{url}"
!fc-cache -f > /dev/null 2>&1
!fc-list | grep -i -E 'anton|bebas|montserrat|oswald|poppins|permanent' | head -10

In [ ]:
# 3. Clone repo Potongan.id + install Python dependencies
!rm -rf /content/potongan
!git clone https://github.com/fransiskusch/potongan.git /content/potongan
%cd /content/potongan
!pip install -q -r backend/requirements.txt uvicorn requests

In [ ]:
# 4. Verifikasi GPU T4 (stop dengan pesan jelas bila bukan T4)
import torch
assert torch.cuda.is_available(), 'GPU TIDAK TERSEDIA. Runtime > Change runtime type > T4 GPU, lalu Run All lagi.'
gpu_name = torch.cuda.get_device_name(0)
print(f'GPU: {gpu_name}')
if 'T4' not in gpu_name:
    print(f'PERINGATAN: GPU {gpu_name} bukan T4 — performa mungkin berbeda, disarankan T4.')

In [ ]:
# 5. Bersihkan file sisa sesi sebelumnya (opsional, aman dijalankan)
import shutil, time, os
for d in ['/content/projects', '/content/uploads']:
    if os.path.isdir(d):
        shutil.rmtree(d, ignore_errors=True)
        print(f'Cleaned: {d}')
print('Local disk siap.')

In [ ]:
#@title 6. Jalankan Backend Potongan.id
CLOUDFLARE_TUNNEL_TOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}
ALLOWED_ORIGINS = "https://clip.fransiskus.my.id" #@param {type:"string"}

import os
os.environ['AUTO_CLIPPER_ALLOWED_ORIGINS'] = ALLOWED_ORIGINS

%cd /content/potongan
!python -m backend.colab_api --cloudflare-token "$CLOUDFLARE_TUNNEL_TOKEN" --api-token "$API_SECRET_TOKEN"